# Urban Air Quality Project: PM2.5 Prediction

This notebook is my data science project about PM2.5 air pollution. I use traffic data and weather data to see how well they can explain and predict PM2.5 levels.

## Research questions

1. Which variables seem most related to PM2.5?
2. Can I predict PM2.5 with regression models?
3. Can I make a simple high-risk / normal-risk classification?
4. Which model works best on this dataset?



## Introduction

I chose this topic because air quality is something that directly affects people's health, especially in big cities. I was curious to see if I could find patterns in pollution levels and understand what factors might influence them.

Also, I wanted to practice working with real-world data and apply some machine learning techniques to a problem that has practical importance.


## Data sources

I used two CSV files. One contains weather and PM2.5 values, and the other contains traffic information. Both files have an hourly `timestamp`, so I can merge them into one dataset.



## Assumptions and success criteria

This is a course project, not an official pollution warning system. I assume that the timestamps match correctly and that the variables are measured for the same hours.

For the project to be successful, I want to show a complete workflow: load the data, check it, merge it, make features, create plots, train models and explain the results.



## Related work and background

Air pollution prediction is usually done with weather variables, traffic data and sometimes more advanced data such as satellite or sensor networks. In this project I use a smaller and simpler dataset, so I focus on basic machine learning models that are easier to understand.

Linear models are useful as a baseline because they are interpretable. Random Forest models can catch more complicated patterns, but they are less transparent. Because of this I compare both types.



## Modeling Approach

After exploring the data, I compare several regression models instead of selecting one manually. Linear Regression, Ridge, Lasso and Random Forest are trained on the same train/test split. The final regression model is selected automatically using the lowest RMSE on the held-out test set.

This is important because Random Forest is not always the best option: if the relationship in the dataset is mostly linear, Linear or Ridge regression can perform better and remain easier to interpret.


In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.model_selection import TimeSeriesSplit, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve,
    precision_recall_curve, average_precision_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report
)

# Locate project root robustly: works when launched from the project root,
# the notebooks/ subfolder, or from VS Code (which sets __vsc_ipynb_file__).
if "__vsc_ipynb_file__" in globals():
    PROJECT_ROOT = Path(globals()["__vsc_ipynb_file__"]).resolve().parent.parent
elif Path.cwd().name == "notebooks":
    PROJECT_ROOT = Path.cwd().parent
elif (Path.cwd() / "requirements.txt").exists():
    PROJECT_ROOT = Path.cwd()
else:
    PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"

# Add src/ to path so analysis_utils can be imported directly
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from analysis_utils import (
    validate_timestamp_key,
    check_hourly_continuity,
    add_time_features,
    make_pm25_risk_label,
    sorted_target_correlations,
)

weather = pd.read_csv(DATA_DIR / "weather_air_quality.csv", parse_dates=["timestamp"])
traffic = pd.read_csv(DATA_DIR / "traffic_counts.csv", parse_dates=["timestamp"])

display(weather.head())
display(traffic.head())
print(weather.shape, traffic.shape)


ModuleNotFoundError: No module named 'analysis_utils'

## Observations

While exploring the data, I noticed that the pollution values change over time rather than across different groups. There are clear fluctuations, which suggests that pollution levels are not constant and may depend on external factors like time or conditions.

I also observed that most values stay within a certain range, with occasional spikes. These spikes might represent unusual events or higher pollution periods, which are important for the model to capture.


## Data validation, merge, and feature engineering

First I check if the timestamp column is usable. Then I merge the two datasets and add a few simple time features.



In [ ]:
# quick check of timestamps in both files
print("Weather:", validate_timestamp_key(weather))
print("Traffic:", validate_timestamp_key(traffic))

# merge only the hours that exist in both files
df = pd.merge(weather, traffic, on="timestamp", how="inner")
df = df.sort_values("timestamp").reset_index(drop=True)

# add_time_features adds hour, day_of_week, and is_weekend from the timestamp column
df = add_time_features(df)

# --- Hourly continuity check ---
# Verifies that every expected hour is present (no skipped hours in the merged series).
cont = check_hourly_continuity(df)
print(f"\nHourly continuity: {cont['missing_hours']} gap(s) detected in {cont['rows']} rows")
if cont["missing_hours"] > 0:
    print("  First gaps at:", cont["gap_timestamps"][:5])

# Threshold set to 25 µg/m³ for this educational project.
# Note: this value is used as a simple hourly proxy for labelling high-risk periods.
# Official PM2.5 standards (e.g. WHO IT-2: 15 µg/m³) are defined as 24-hour averages,
# not per-hour limits, so this threshold should NOT be interpreted as a regulatory limit.
PM25_RISK_THRESHOLD = 25
df = make_pm25_risk_label(df, threshold=PM25_RISK_THRESHOLD)

# --- Lag feature: previous hour's PM2.5 ---
# PM2.5 is strongly autocorrelated: the level at hour t is a good predictor of hour t+1.
# The first row becomes NaN after shifting and is dropped to keep the data clean.
df["pm25_lag_1h"] = df["pm25_ugm3"].shift(1)
df = df.dropna(subset=["pm25_lag_1h"]).reset_index(drop=True)

print("\nShape after adding pm25_lag_1h (first row dropped):", df.shape)
display(df.isna().sum())
display(df.describe())
print("High-risk share:", round(df["high_pm25_risk"].mean(), 3))


Weather: {'rows': 2160, 'missing_timestamps': 0, 'duplicate_timestamps': 0}
Traffic: {'rows': 2160, 'missing_timestamps': 0, 'duplicate_timestamps': 0}

Hourly continuity: 0 gap(s) detected in 2160 rows

Shape after adding pm25_lag_1h (first row dropped): (2159, 13)


timestamp           0
temperature_c       0
humidity_pct        0
wind_speed_ms       0
pm25_ugm3           0
traffic_volume      0
avg_speed_kmh       0
congestion_index    0
hour                0
day_of_week         0
is_weekend          0
high_pm25_risk      0
pm25_lag_1h         0
dtype: int64

,timestamp,temperature_c,humidity_pct,wind_speed_ms,pm25_ugm3,traffic_volume,avg_speed_kmh,congestion_index,hour,day_of_week,is_weekend,high_pm25_risk,pm25_lag_1h
count,2159,2159.000000,2159.000000,2159.000000,2159.000000,2159.000000,2159.000000,2159.000000,2159.000000,2159.000000,2159.000000,2159.000000,2159.000000
mean,2026-02-15 00:00:00,9.443446,54.218666,3.055660,17.796572,447.431681,36.431172,0.310059,11.505327,3.011116,0.289023,0.115794,17.795484
min,2026-01-01 01:00:00,-4.900000,25.000000,1.340000,3.000000,80.000000,14.100000,0.000000,0.000000,0.000000,0.000000,0.000000,3.000000
25%,2026-01-23 12:30:00,5.800000,43.700000,2.290000,13.590000,344.000000,32.100000,0.144500,6.000000,1.000000,0.000000,0.000000,13.590000
50%,2026-02-15 00:00:00,9.300000,54.200000,3.070000,17.350000,429.000000,36.700000,0.275000,12.000000,3.000000,0.000000,0.000000,17.340000
75%,2026-03-09 11:30:00,13.200000,65.000000,3.650000,21.635000,529.000000,40.700000,0.441000,17.500000,5.000000,1.000000,0.000000,21.635000
max,2026-03-31 23:00:00,21.600000,98.000000,7.380000,37.080000,924.000000,54.600000,1.000000,23.000000,6.000000,1.000000,1.000000,37.080000
std,NaN,5.149537,13.601969,0.943408,5.820658,146.927462,6.455080,0.221828,6.920965,2.009216,0.453413,0.320052,5.820723


High-risk share: 0.116


### Validation notes

This step checks missing timestamps, duplicate timestamps, missing values after merging, and hourly continuity (no skipped hours). I included it because problems here could affect all the modelling later.


In [ ]:
# small validation table
vk_weather = validate_timestamp_key(weather)
vk_traffic = validate_timestamp_key(traffic)
vk_merged  = validate_timestamp_key(df)

cont_weather = check_hourly_continuity(weather)
cont_traffic = check_hourly_continuity(traffic)
cont_merged  = check_hourly_continuity(df)

validation_report = pd.DataFrame({
    "dataset": ["weather_air_quality", "traffic_counts", "merged"],
    "rows": [vk_weather["rows"], vk_traffic["rows"], vk_merged["rows"]],
    "duplicate_timestamps": [
        vk_weather["duplicate_timestamps"],
        vk_traffic["duplicate_timestamps"],
        vk_merged["duplicate_timestamps"],
    ],
    "hourly_gaps": [
        cont_weather["missing_hours"],
        cont_traffic["missing_hours"],
        cont_merged["missing_hours"],
    ],
    "total_missing_values": [
        weather.isna().sum().sum(),
        traffic.isna().sum().sum(),
        df.isna().sum().sum(),
    ],
})

display(validation_report)


,dataset,rows,duplicate_timestamps,hourly_gaps,total_missing_values
0,weather_air_quality,2160,0,0,0
1,traffic_counts,2160,0,0,0
2,merged,2159,0,0,0
